In [1]:
import os, re, glob, math, time, random
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageOps
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm

import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score
from tqdm import tqdm
from ultralytics import YOLO
import cv2

In [2]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
seed_everything(42)

In [3]:
DATA_DIR = Path(r"E:\multiview_pig_posture_recognition")
print("DATA_DIR:", DATA_DIR)
train_csv = DATA_DIR / (f"train1.csv")
train1_img_dir = DATA_DIR / "train1_images"

DATA_DIR: E:\multiview_pig_posture_recognition


Use photos taken in 20250209 as test

In [4]:
train_df = pd.read_csv(train_csv)
mask_test = train_df["image_id"].str.contains(r"_20250209_")
test_df = train_df.loc[mask_test].copy()
train_df = train_df.loc[~mask_test].copy()

In [5]:
train_df = train_df.sort_values("image_id")
classes_txt = DATA_DIR / "pig_posture_classes.txt"
with open(classes_txt, "r") as f:
        CLASS_NAMES = [line.strip() for line in f if line.strip()]
pd.DataFrame({
    "label": range(5),
    "class_name": CLASS_NAMES
})

,label,class_name
0,0,Lateral_lying_left
1,1,Lateral_lying_right
2,2,Sitting
3,3,Standing
4,4,Sternal_lying


In [6]:
def parse_cam_type(image_id: str):
    m = re.search(r"_([a-zA-Z]+)_cam\d+_", str(image_id))
    return m.group(1).lower() if m else "unk"
train_df["cam_type"] = train_df["image_id"].map(parse_cam_type)
test_df["cam_type"]  = test_df["image_id"].map(parse_cam_type)

In [7]:
def parse_bbox_string(bbox_str):
    s = str(bbox_str).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1].strip()
    s = s.replace(",", " ")
    parts = [p for p in s.split() if p]
    if len(parts) != 4:
        raise ValueError(f"Bad bbox: {bbox_str}")
    x, y, w, h = map(float, parts)
    return x, y, w, h

bx = train_df["bbox"].apply(parse_bbox_string)
train_df["x"] = bx.apply(lambda t: t[0])
train_df["y"] = bx.apply(lambda t: t[1])
train_df["w"] = bx.apply(lambda t: t[2])
train_df["h"] = bx.apply(lambda t: t[3])

bx = test_df["bbox"].apply(parse_bbox_string)
test_df["x"] = bx.apply(lambda t: t[0])
test_df["y"] = bx.apply(lambda t: t[1])
test_df["w"] = bx.apply(lambda t: t[2])
test_df["h"] = bx.apply(lambda t: t[3])

In [8]:
def parse_scene(image_id):
    parts = image_id.split("_")

    pen  = parts[0]        # pen1/pen2
    date = parts[3]        # 20250108 date

    return f"{pen}_{date}"

To avoid data leakage caused by short time window (e.g., two pictures taken in 10s successively), Any two images, provided that the time difference is less than 45 seconds and are both taken by the same camera , must be entered into the train or the val as a whole and cannot be separated.

In [9]:
train_df["datetime"] = pd.to_datetime(
    train_df["image_id"].str.extract(r'(\d{8}_\d{6})')[0],
    format="%Y%m%d_%H%M%S"
)
train_df = train_df.sort_values(["cam_type", "datetime"]).reset_index(drop=True)
time_threshold = 45  # time window
group_ids = []
current_group = 0

for cam in train_df["cam_type"].unique():

    cam_idx = train_df["cam_type"] == cam
    cam_df = train_df[cam_idx]

    last_time = None

    for idx in cam_df.index:
        current_time = train_df.loc[idx, "datetime"]
        if last_time is None:
            group_ids.append(current_group)
        else:
            diff = (current_time - last_time).total_seconds()
            if diff <= time_threshold:
                group_ids.append(current_group)
            else:
                current_group += 1
                group_ids.append(current_group)

        last_time = current_time

    current_group += 1

train_df["time_group"] = group_ids

In [10]:
img_dom = (
    train_df
    .groupby("image_id") # avoid data leakage, separate pictures, instead of patch into train/test
    .agg(dom_label=("class_id", lambda s: s.value_counts().idxmax()), # select the largest class group to represent the picture
         cam_type=("cam_type", "first"), # a picture is taken by a specific angle, which means that the camera type should be identical
            time_group=("time_group", "first"))
    .reset_index()
)

img_dom["scene_time_group"] = (
    img_dom["cam_type"].astype(str)
    + "_"
    + img_dom["time_group"].astype(str)
)

In [11]:
from sklearn.model_selection import StratifiedGroupKFold
train_df_new = []
val_df_new =[]
sgkf = StratifiedGroupKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

splits = list(
    sgkf.split(
        img_dom,
        y=img_dom["dom_label"], # This tells the splitter to maintain the class distribution of dom_label across the folds.
        groups=img_dom["scene_time_group"] # all samples belonging to the same group are assigned to the same fold.
    )
)

train_idx, val_idx = splits[0]

train_imgs = set(img_dom.iloc[train_idx]["image_id"])
val_imgs   = set(img_dom.iloc[val_idx]["image_id"])

train_df_new = train_df[train_df["image_id"].isin(train_imgs)].reset_index(drop=True)
val_df_new   = train_df[train_df["image_id"].isin(val_imgs)].reset_index(drop=True)

E:\multiview_pig_posture_recognition\.venv3\lib\site-packages\sklearn\model_selection\_split.py:1035: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(


In [12]:
print("train:", train_df_new.shape, "test:", test_df.shape, "val:", val_df_new.shape)

train: (16678, 13) test: (1744, 11) val: (4512, 13)


In [13]:
def resolve_train_image_path(image_id: str):
    p1 = train1_img_dir / image_id
    if p1.exists(): return p1
    hits = glob.glob(str(DATA_DIR / "**" / str(image_id)), recursive=True)
    if hits: return Path(hits[0])
    raise FileNotFoundError(f"Train image not found: {image_id}")

def resolve_test_image_path(image_id: str):
    p = train1_img_dir / image_id
    if p.exists(): return p
    hits = glob.glob(str(DATA_DIR / "**" / str(image_id)), recursive=True)
    if hits: return Path(hits[0])
    raise FileNotFoundError(f"Test image not found: {image_id}")

In [14]:
train_df_new.head()

,row_id,image_id,width,height,bbox,class_id,cam_type,x,y,w,h,datetime,time_group
0,train_pen1_orb_cam2_20250108_080108_0004,pen1_orb_cam2_20250108_080108.jpg,1920,1080,"[817.7,0.0,262.0,264.3]",3,orb,817.7,0.0,262.0,264.3,2025-01-08 08:01:08,0
1,train_pen1_orb_cam2_20250108_080108_0005,pen1_orb_cam2_20250108_080108.jpg,1920,1080,"[428.0,416.0,243.0,433.0]",4,orb,428.0,416.0,243.0,433.0,2025-01-08 08:01:08,0
2,train_pen1_orb_cam2_20250108_080108_0002,pen1_orb_cam2_20250108_080108.jpg,1920,1080,"[1204.5,4.0,209.0,296.0]",3,orb,1204.5,4.0,209.0,296.0,2025-01-08 08:01:08,0
3,train_pen1_orb_cam2_20250108_080108_0000,pen1_orb_cam2_20250108_080108.jpg,1920,1080,"[987.0,314.5,447.0,472.0]",3,orb,987.0,314.5,447.0,472.0,2025-01-08 08:01:08,0
4,train_pen1_orb_cam2_20250108_080108_0001,pen1_orb_cam2_20250108_080108.jpg,1920,1080,"[559.5,199.0,296.0,417.0]",3,orb,559.5,199.0,296.0,417.0,2025-01-08 08:01:08,0


# pig  bbox regression (yolo)

## yolo

In [18]:
def convert_to_yolo(df):

    df = df.copy()

    df["x_center"] = (df["x"] + df["w"]/2) / df["width"]
    df["y_center"] = (df["y"] + df["h"]/2) / df["height"]
    df["w_norm"] = df["w"] / df["width"]
    df["h_norm"] = df["h"] / df["height"]

    return df

In [19]:
def write_yolo_labels(df, label_dir):

    Path(label_dir).mkdir(parents=True, exist_ok=True)

    for img_id, group in df.groupby("image_id"):

        label_path = Path(label_dir) / img_id.replace(".jpg", ".txt")

        with open(label_path, "w") as f:
            for _, row in group.iterrows():

                line = f"{row.class_id} {row.x_center} {row.y_center} {row.w_norm} {row.h_norm}\n"
                f.write(line)

In [21]:
import shutil
def copy_images(df, src_dir, dst_dir):

    Path(dst_dir).mkdir(parents=True, exist_ok=True)

    for img in df["image_id"].unique():

        src = src_dir / img
        dst = Path(dst_dir) / img

        if src.exists():
            shutil.copy(src, dst)

In [22]:
train_df = convert_to_yolo(train_df_new)
val_df = convert_to_yolo(val_df_new)
test_df = convert_to_yolo(test_df)

write_yolo_labels(train_df, "dataset/labels/train")
write_yolo_labels(val_df, "dataset/labels/val")
write_yolo_labels(test_df, "dataset/labels/test")

copy_images(train_df, train1_img_dir, "dataset/images/train")
copy_images(val_df, train1_img_dir, "dataset/images/val")
copy_images(test_df, train1_img_dir, "dataset/images/test")

AdamW(lr=0.001111, momentum=0.9)
Let yolo decide the best parameter

In [2]:
model = YOLO("yolov8n.pt")

model.train(
    data="data.yaml",
    epochs=10,
    imgsz=640,
    name="yolo_exp16"
)

Ultralytics 8.4.19  Python-3.10.11 torch-2.12.0.dev20260224+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_exp16, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000020AC081C730>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
    

## SSD

In [15]:
train_df_new["xmin"] = train_df_new["x"]
train_df_new["ymin"] = train_df_new["y"]
train_df_new["xmax"] = train_df_new["x"] + train_df_new["w"]
train_df_new["ymax"] = train_df_new["y"] + train_df_new["h"]

In [16]:
test_df["xmin"] = test_df["x"]
test_df["ymin"] = test_df["y"]
test_df["xmax"] = test_df["x"] + test_df["w"]
test_df["ymax"] = test_df["y"] + test_df["h"]

In [17]:
val_df_new["xmin"] = val_df_new["x"]
val_df_new["ymin"] = val_df_new["y"]
val_df_new["xmax"] = val_df_new["x"] + val_df_new["w"]
val_df_new["ymax"] = val_df_new["y"] + val_df_new["h"]

In [18]:
import torchvision
class PigDataset(torch.utils.data.Dataset):

    def __init__(self, df, img_dir, transforms=None):
        self.df = df
        self.img_dir = img_dir
        self.transforms = transforms
        self.image_ids = df.image_id.unique()

    def __getitem__(self, idx):

        image_id = self.image_ids[idx]
        records = self.df[self.df.image_id == image_id]

        img_path = os.path.join(self.img_dir, image_id)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        boxes = records[["xmin","ymin","xmax","ymax"]].values
        labels = records["class_id"].values + 1

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        if self.transforms:
            img = self.transforms(img)
        else:
            img = torchvision.transforms.ToTensor()(img)

        return img, target

    def __len__(self):
        return len(self.image_ids)

In [19]:
num_classes = 6   # 5 posture + background

model = torchvision.models.detection.ssd300_vgg16(
    weights="DEFAULT"
)

model.head.classification_head.num_classes = num_classes

Downloading: "https://download.pytorch.org/models/ssd300_vgg16_coco-b556d3b4.pth" to C:\Users\chenk/.cache\torch\hub\checkpoints\ssd300_vgg16_coco-b556d3b4.pth


100%|██████████| 136M/136M [00:02<00:00, 62.0MB/s] 


In [26]:
train_dataset = PigDataset(
    train_df_new,
    train1_img_dir
)
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=lambda x: tuple(zip(*x))
)

In [27]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [28]:
params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    params,
    lr=0.001111,
    weight_decay=0.0005
)

In [29]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for images, targets in progress_bar:
        images = [img.to(device) for img in images]
        targets = [
            {k: v.to(device) for k, v in t.items()}
            for t in targets
        ]
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        progress_bar.set_postfix(
            loss=loss.item()
        )
    print("Epoch loss:", epoch_loss / len(train_loader))

Epoch 1/5: 100%|██████████| 585/585 [02:55<00:00,  3.33it/s, loss=5.17]


Epoch loss: 6.215165045322516


Epoch 2/5: 100%|██████████| 585/585 [01:39<00:00,  5.89it/s, loss=3.15]


Epoch loss: 4.511548455556234


Epoch 3/5: 100%|██████████| 585/585 [01:36<00:00,  6.05it/s, loss=2.62]


Epoch loss: 3.9766503574501755


Epoch 4/5: 100%|██████████| 585/585 [01:38<00:00,  5.92it/s, loss=3.12]


Epoch loss: 3.45458534089928


Epoch 5/5: 100%|██████████| 585/585 [01:36<00:00,  6.08it/s, loss=3.34] 

Epoch loss: 3.0084034852492505


In [30]:
torch.save(model.state_dict(), "ssd_pig_model.pth")

# IoU（Intersection over Union）

## SSD

In [67]:
model = torchvision.models.detection.ssd300_vgg16(weights=None)
model.load_state_dict(torch.load("ssd_pig_model.pth"))

model = model.to(device)
model.eval()

SSD(
  (backbone): SSDFeatureExtractorVGG(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (15): ReLU(inplace=

In [71]:
import cv2
image_id = test_df["image_id"].iloc[0]
print(image_id)
img_path = train1_img_dir / image_id
img = cv2.imread(str(img_path))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

img_tensor = torch.from_numpy(img).permute(2,0,1).float()/255
img_tensor = img_tensor.to(device)

model.eval()

with torch.no_grad():
    outputs = model([img_tensor])

pred_boxes = outputs[0]["boxes"].cpu().numpy()
scores = outputs[0]["scores"].cpu().numpy()
labels = outputs[0]["labels"].cpu().numpy()

pen2_tur_cam1_20250209_070148.jpg


In [72]:
pred_boxes = outputs[0]["boxes"].cpu().numpy()
scores = outputs[0]["scores"].cpu().numpy()
img_gt = img.copy()

for _, row in gt.iterrows():

    x1 = int(row["x"])
    y1 = int(row["y"])
    x2 = int(row["x"] + row["w"])
    y2 = int(row["y"] + row["h"])

    cv2.rectangle(
        img_gt,
        (x1, y1),
        (x2, y2),
        (0,255,0),   # green
        2
    )
img_pred = img_gt.copy()

for box, score in zip(pred_boxes, scores):

    if score < 0.5:
        continue

    x1, y1, x2, y2 = map(int, box)

    cv2.rectangle(
        img_pred,
        (x1, y1),
        (x2, y2),
        (255,0,0),  # red
        2
    )

In [73]:
from matplotlib.patches import Patch

plt.figure(figsize=(8,6))

plt.imshow(img_pred)
plt.axis("off")

legend_elements = [
    Patch(facecolor='none', edgecolor='green', label='Ground Truth'),
    Patch(facecolor='none', edgecolor='red', label='SSD Prediction')
]

plt.legend(
    handles=legend_elements,
    loc="upper right",
    frameon=True
)

plt.show()

<IPython.core.display.Javascript object>

In [74]:
def iou_xyxy(boxA, boxB):

    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter = inter_w * inter_h

    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    union = areaA + areaB - inter

    return inter / union if union > 0 else 0

In [75]:
gt_boxes = []

for _, row in gt.iterrows():

    x1 = row["x"]
    y1 = row["y"]
    x2 = row["x"] + row["w"]
    y2 = row["y"] + row["h"]

    gt_boxes.append([x1, y1, x2, y2])

In [91]:
score_threshold = 0.5

filtered_pred_boxes = []
for box, score in zip(pred_boxes, scores):
    if score >= score_threshold:
        filtered_pred_boxes.append(box)

In [92]:
ious = []

for pbox in filtered_pred_boxes:
    for gtbox in gt_boxes:

        iou = iou_xyxy(pbox, gtbox)
        ious.append(iou)

In [93]:
import numpy as np
mean_iou = np.mean(ious)
print("Mean IoU:", mean_iou)

Mean IoU: 0.07890809


## Yolo

In [62]:
%matplotlib notebook

In [46]:
image_id = test_df["image_id"].iloc[0]
print(image_id)

pen2_tur_cam1_20250209_070148.jpg


use one image as example

In [47]:
import cv2
import matplotlib.pyplot as plt

img_path = train1_img_dir / image_id

img = cv2.imread(str(img_path))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

In [48]:
gt = test_df[test_df["image_id"] == image_id]

In [49]:
img_gt = img.copy()

for _, row in gt.iterrows():

    x1 = int(row["x"])
    y1 = int(row["y"])
    x2 = int(row["x"] + row["w"])
    y2 = int(row["y"] + row["h"])

    cv2.rectangle(
        img_gt,
        (x1, y1),
        (x2, y2),
        (0,255,0),   # green
        2
    )

In [50]:
model = YOLO("runs/detect/yolo_exp16/weights/best.pt")

In [51]:
results = model(str(img_path))


image 1/1 E:\multiview_pig_posture_recognition\train1_images\pen2_tur_cam1_20250209_070148.jpg: 384x640 8 Standings, 1 Sternal_lying, 583.5ms
Speed: 2.4ms preprocess, 583.5ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)


In [52]:
pred_boxes = results[0].boxes.xyxy.cpu().numpy()
img_pred = img_gt.copy()

for box in pred_boxes:

    x1, y1, x2, y2 = map(int, box)

    cv2.rectangle(
        img_pred,
        (x1,y1),
        (x2,y2),
        (255,0,0),  # red
        2
    )

In [53]:
from matplotlib.patches import Patch
fig = plt.figure()

plt.imshow(img_pred)
plt.axis("off")

legend_elements = [
    Patch(facecolor='none', edgecolor='green', label='Ground Truth'),
    Patch(facecolor='none', edgecolor='red', label='YOLO Prediction')
]

plt.legend(
    handles=legend_elements,
    loc="upper right",
    frameon=True
)

plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.show()

<IPython.core.display.Javascript object>

In [54]:
def compute_iou(box1, box2):

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2-x1) * max(0, y2-y1)

    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union = area1 + area2 - inter

    if union == 0:
        return 0

    return inter / union

In [28]:

image_ids = test_df["image_id"].unique()
ious = []

In [29]:
for image_id in image_ids:
    img_path = train1_img_dir / image_id
    results = model(str(img_path))
    pred_boxes = results[0].boxes.xyxy.cpu().numpy()


image 1/1 E:\multiview_pig_posture_recognition\train1_images\pen2_tur_cam1_20250209_070148.jpg: 384x640 8 Standings, 1 Sternal_lying, 7.8ms
Speed: 2.3ms preprocess, 7.8ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 E:\multiview_pig_posture_recognition\train1_images\pen2_tur_cam1_20250209_070508.jpg: 384x640 7 Standings, 2 Sternal_lyings, 7.3ms
Speed: 1.8ms preprocess, 7.3ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 E:\multiview_pig_posture_recognition\train1_images\pen2_tur_cam1_20250209_070648.jpg: 384x640 1 Lateral_lying_left, 8 Standings, 1 Sternal_lying, 8.9ms
Speed: 1.8ms preprocess, 8.9ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 E:\multiview_pig_posture_recognition\train1_images\pen2_tur_cam1_20250209_071008.jpg: 384x640 1 Lateral_lying_left, 1 Lateral_lying_right, 6 Standings, 3 Sternal_lyings, 8.6ms
Speed: 1.6ms preprocess, 8.6ms inference, 2.9ms postprocess per image at shape 

In [30]:
    gt_df = test_df[test_df["image_id"] == image_id]

    gt_boxes = []

    for _,row in gt_df.iterrows():

        x1 = row["x"]
        y1 = row["y"]
        x2 = row["x"] + row["w"]
        y2 = row["y"] + row["h"]

        gt_boxes.append([x1,y1,x2,y2])

In [31]:
    for pred in pred_boxes:
        best_iou = 0
        for gt in gt_boxes:
            iou = compute_iou(pred, gt)
            best_iou = max(best_iou, iou)
        ious.append(best_iou)

In [32]:
import numpy as np
mean_iou = np.mean(ious)
print("Mean IoU:", mean_iou)

Mean IoU: 0.8747141
